# 18 - Test Packed Recovered L78

Notebook de validação pós-marco para o caminho packed/materialized L78.

Objetivo: carregar `student_pruned`, materializar `mlp_ternary_2bit.bin`, aplicar o adapter `student_packed_ternary_recovered_l78_t4_resume` e testar novamente o modelo recuperado sem retreino.

Marco anterior: `17_t4_resume_packed_recovery` fechou `100/100` (`1.00`) contra referência limpa `0.92` no benchmark local de 100 perguntas.

In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os, sys, json, gc, re, time, hashlib

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
PACKED_RECOVERY_RESULTS_DIR = RESULTS_DIR / 'packed_recovery'
for p in [DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, DEPLOY_DIR, EXPORTS_DIR, PACKED_RECOVERY_RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)
print('PACKED_RECOVERY_RESULTS_DIR =', PACKED_RECOVERY_RESULTS_DIR)

In [ ]:
# Dependências de validação. Não instala torchao: versões antigas podem quebrar PEFT/LoRA no Colab.
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-U',
    'transformers',
    'accelerate',
    'sentencepiece',
    'safetensors',
    'peft',
])

In [ ]:
import numpy as np
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

HF_TOKEN = os.environ.get('HF_TOKEN')
PRUNED_CHECKPOINT = CHECKPOINTS_DIR / 'student_pruned'
PACKED_RECOVERY_ADAPTER = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_t4_resume'
QUESTIONS_PATH = DATA_DIR / 'gemma4_100_questions.json'
EXPECTED_PACKED_SHA256 = '0b6849fe3ceaab615a4e54f63f5a96e9d9d431ba03b7b791a7ab0638ea174116'
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'
MAX_NEW_TOKENS = 24

PACKAGE_CANDIDATES = [
    EXPORTS_DIR / 'ternary_mlp_l78',
    DEPLOY_DIR,
    PROJECT_ROOT / 'exports' / 'ternary_mlp_l78',
    PROJECT_ROOT / 'deploy',
]
PACKAGE_DIR = next((p for p in PACKAGE_CANDIDATES if (p / 'manifest.json').exists() and (p / 'mlp_ternary_2bit.bin').exists()), None)
if PACKAGE_DIR is None:
    raise FileNotFoundError('Pacote packed não encontrado em: ' + json.dumps([str(p) for p in PACKAGE_CANDIDATES], ensure_ascii=False, indent=2))

MANIFEST_PATH = PACKAGE_DIR / 'manifest.json'
BIN_PATH = PACKAGE_DIR / 'mlp_ternary_2bit.bin'

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

manifest = load_json(MANIFEST_PATH)
questions = load_json(QUESTIONS_PATH)
packed_sha256 = sha256_file(BIN_PATH)
assert manifest.get('format') == 'rpt_hybrid_ternary_mlp_lora_v1', manifest.get('format')
assert manifest['summary']['num_tensors'] == 78, manifest['summary']
assert packed_sha256 == EXPECTED_PACKED_SHA256, packed_sha256
assert PRUNED_CHECKPOINT.exists(), PRUNED_CHECKPOINT
assert PACKED_RECOVERY_ADAPTER.exists(), PACKED_RECOVERY_ADAPTER
assert QUESTIONS_PATH.exists(), QUESTIONS_PATH

print('PACKAGE_DIR =', PACKAGE_DIR)
print('packed_sha256 =', packed_sha256)
print('questions =', len(questions))
print('adapter =', PACKED_RECOVERY_ADAPTER)

In [ ]:
# 2-bit unpack e materialização dos 78 MLPs no student_pruned.
def unpack_codes_from_blob(blob, n):
    raw = np.frombuffer(blob, dtype=np.uint8)
    codes = np.empty(raw.size * 4, dtype=np.uint8)
    codes[0::4] = raw & 0b11
    codes[1::4] = (raw >> 2) & 0b11
    codes[2::4] = (raw >> 4) & 0b11
    codes[3::4] = (raw >> 6) & 0b11
    return codes[:n]

def unpack_tensor_from_blob(blob, n, shape, scale, dtype=torch.float32):
    codes = unpack_codes_from_blob(blob, n)
    vals = np.zeros(n, dtype=np.float32)
    vals[codes == 1] = scale
    vals[codes == 2] = -scale
    return torch.from_numpy(vals).view(tuple(shape)).to(dtype=dtype)

def materialize_packed_mlp_into_model(model, manifest, bin_path):
    named_params = dict(model.named_parameters())
    missing_params = []
    meta_params = []
    applied = []
    with open(bin_path, 'rb') as f:
        for item in manifest['tensors']:
            name = item['name']
            param = named_params.get(name)
            if param is None:
                missing_params.append(name)
                continue
            if getattr(param, 'is_meta', False):
                meta_params.append(name)
                continue
            f.seek(item['offset'])
            blob = f.read(item['packed_bytes'])
            recon = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'], dtype=torch.float32)
            if tuple(param.shape) != tuple(recon.shape):
                raise RuntimeError(f'Shape mismatch for {name}: param={tuple(param.shape)} recon={tuple(recon.shape)}')
            with torch.no_grad():
                param.copy_(recon.to(device=param.device, dtype=param.dtype))
            applied.append(name)
            del recon
    if missing_params or meta_params:
        raise RuntimeError(json.dumps({'missing_params': missing_params, 'meta_params': meta_params}, ensure_ascii=False, indent=2))
    if len(applied) != 78:
        raise RuntimeError(f'Expected 78 MLP tensors applied, got {len(applied)}')
    return applied

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def normalize_text(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())

def judge_answer(question_payload, answer):
    actual = normalize_text(answer)
    accepted = [question_payload.get('expected_answer', '')]
    accepted.extend(question_payload.get('accepted_answers') or [])
    for candidate in accepted:
        expected = normalize_text(candidate)
        if expected and (expected in actual or actual in expected):
            return True
    return False

def extract_answer(prompt, generated):
    text = generated
    if prompt in text:
        text = text.split(prompt, 1)[-1]
    return text.strip().split('\n')[0].strip()

def run_eval(model, tokenizer, questions, max_new_tokens=MAX_NEW_TOKENS, verbose=True):
    rows = []
    correct = 0
    device = next(model.parameters()).device
    for idx, q in enumerate(questions):
        prompt = PROMPT_TEMPLATE.format(question=q['question'])
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(out[0], skip_special_tokens=True)
        answer = extract_answer(prompt, generated)
        ok = judge_answer(q, answer)
        correct += int(ok)
        row = {
            'id': q.get('id', idx + 1),
            'category': q.get('category'),
            'question': q['question'],
            'expected_answer': q['expected_answer'],
            'model_answer': answer,
            'is_correct': ok,
            'raw_output': generated,
        }
        rows.append(row)
        if verbose:
            print(f"{idx + 1:03d}/{len(questions)}", 'OK' if ok else 'ERR', '|', q['question'], '=>', answer)
    by_category = {}
    for row in rows:
        cat = row.get('category') or 'unknown'
        by_category.setdefault(cat, {'total': 0, 'correct': 0})
        by_category[cat]['total'] += 1
        by_category[cat]['correct'] += int(row['is_correct'])
    for vals in by_category.values():
        vals['accuracy'] = vals['correct'] / vals['total'] if vals['total'] else 0.0
    summary = {
        'total_questions': len(questions),
        'correct': correct,
        'accuracy': correct / len(questions) if questions else 0.0,
        'by_category': by_category,
    }
    return rows, summary

In [ ]:
# Carrega o modelo final: student_pruned -> packed MLP materializado -> adapter recuperado.
cleanup_memory()
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

tokenizer = AutoTokenizer.from_pretrained(str(PRUNED_CHECKPOINT), token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    str(PRUNED_CHECKPOINT),
    dtype='auto',
    device_map='auto',
    token=HF_TOKEN,
)
applied = materialize_packed_mlp_into_model(base_model, manifest, BIN_PATH)
model = PeftModel.from_pretrained(base_model, str(PACKED_RECOVERY_ADAPTER), token=HF_TOKEN, is_trainable=False)
model.eval()
print('applied_packed_mlp_tensors =', len(applied))
print('model ready')

In [ ]:
# Reexecuta o benchmark local de 100 perguntas e salva relatório timestampado.
start = time.time()
rows, summary = run_eval(model, tokenizer, questions, verbose=True)
elapsed_sec = round(time.time() - start, 3)

report = {
    'notebook': '18_test_packed_recovered_l78',
    'purpose': 'Post-milestone validation of packed/materialized recovered L78 model.',
    'load_order': 'student_pruned -> materialize mlp_ternary_2bit.bin -> PeftModel.from_pretrained(recovered adapter)',
    'adapter_path': str(PACKED_RECOVERY_ADAPTER),
    'manifest_path': str(MANIFEST_PATH),
    'bin_path': str(BIN_PATH),
    'packed_bin_sha256': packed_sha256,
    'applied_packed_mlp_tensors': len(applied),
    'summary': summary,
    'rows': rows,
    'baselines': {
        'packed_base_no_adapter_accuracy': 0.01,
        'old_l78_on_packed_accuracy': 0.63,
        'clean_base_plus_l78_reference_accuracy': 0.92,
        'hf_gemma_2_2b_it_original_reference_accuracy': 0.90,
    },
    'elapsed_sec': elapsed_sec,
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
}
out_path = PACKED_RECOVERY_RESULTS_DIR / f"18_test_packed_recovered_l78_{time.strftime('%Y%m%d_%H%M%S')}.json"
save_json(report, out_path)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('saved:', out_path)

In [ ]:
# Smoke test opcional fora da bateria oficial. Não conta como métrica científica; serve para inspeção qualitativa.
extra_questions = [
    'Explique em uma frase por que água ferve ao nível do mar perto de 100 graus Celsius.',
    'Se um trem percorre 120 km em 2 horas, qual é a velocidade média?',
    'Escreva uma função Python curta que some dois números.',
    'Qual é a diferença entre pruning e quantização em modelos de linguagem?',
]

device = next(model.parameters()).device
for question in extra_questions:
    prompt = PROMPT_TEMPLATE.format(question=question)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(out[0], skip_special_tokens=True)
    print('\nPERGUNTA:', question)
    print('RESPOSTA:', extract_answer(prompt, generated))

## Próximos passos recomendados

1. Rodar este notebook em Colab com GPU e confirmar que o resultado reproduz `100/100` sem retreino.
2. Criar uma bateria maior e inédita para reduzir risco de overfit no conjunto de 100 perguntas.
3. Comparar o mesmo checkpoint contra `student_pruned`, `HF Gemma 2B original`, `old_l78_on_packed` e `clean_base + l78` em protocolo único.
4. Medir tamanho, latência e uso de VRAM/RAM do caminho packed/materialized.
5. Só depois decidir se vale avançar para attention ou para runtime/export mais agressivo.